In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model
from sklearn.metrics import mean_absolute_error

# Fonts and tick marks and such
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Georgia', 'Times New Roman', 'DejaVu Serif'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'grid.linewidth': 0.6,
    'figure.facecolor': '#FAFAF8',
    'axes.facecolor': '#FAFAF8',
    'axes.labelsize'  : 14,
    'axes.titlesize'  : 15,
    'xtick.labelsize' : 12,
    'ytick.labelsize' : 12,
    'legend.fontsize' : 12,
    'figure.titlesize': 16,
})

# Colors
LIGHT = '#FAFAF8'
LIGHT = '#FAFAF8'
NAVY = '#2C3E6B'
RED  = '#C0392B'

# Labels for variables (full name and measurement scale)
VAR_LABELS = {
    'tasmax'            : 'Max Temp (°C)',
    'tasmin'            : 'Min Temp (°C)',
    'rsds'              : 'Solar Radiation (W/m²)',
    'sfcWind'           : 'Wind Speed (m/s)',
    'ps'                : 'Surface Pressure (Pa)',
    'pr'                : 'Precipitation (kg/m²/s)',
    'Anxiety Disorders' : 'Anxiety Disorders',
}

# Default Scikit-learn values for params
DEFAULT_ALPHA_1  = 1e-6
DEFAULT_ALPHA_2  = 1e-6
DEFAULT_LAMBDA_1 = 1e-6
DEFAULT_LAMBDA_2 = 1e-6
DEFAULT_N_ITER   = 300
DEFAULT_TOL      = 1e-3

# Load and prep weather data
weather = pd.read_csv('San_Francisco_climate_v2.csv')
# Split date (called time) into year and month
weather['time'] = pd.to_datetime(weather['time'])
weather['year']  = weather['time'].dt.year
weather['month'] = weather['time'].dt.month
weather = weather.groupby(['year', 'month']).mean().reset_index()
weather = weather.drop(columns=['time'], errors='ignore')
weather = weather.drop(columns=['time'], errors='ignore')
# Make months cyclical so that the model understands the order
# of months e.g., Jan comes after Dec and Feb comes after Jan
# (aka they aren't just numbers 1-12 that repeat anymore)
weather['month_sin'] = np.sin(2 * np.pi * weather['month'] / 12)
weather['month_cos'] = np.cos(2 * np.pi * weather['month'] / 12)

# Load and prep mental health data to get anxiety
mental = pd.read_csv('mental_health_ED.csv')
ANXIETY_COL = 'Anxiety Disorders'
# Only want the total values of anxiety for each month (All)
mental = mental[mental['demographics_values'] == 'All']
mental = mental[mental['condition'] == 'Anxiety Disorders']
mental['month_end'] = pd.to_datetime(mental['month_end'])
# Pivot so that it is in same format as weather data
mental_pivot = mental.pivot_table(
    index='month_end',
    values='rate_per_100000_visits'
).reset_index()
mental_pivot.columns = ['month_end', ANXIETY_COL]
# Split date (called month_end) into year and month
mental_pivot['month_end'] = pd.to_datetime(mental_pivot['month_end'])
mental_pivot['year']  = mental_pivot['month_end'].dt.year
mental_pivot['month'] = mental_pivot['month_end'].dt.month
mental_pivot = mental_pivot.drop(columns='month_end')

# Merge into one data set
combined = weather.merge(mental_pivot, on=['year', 'month'], how='inner')

# Features are cyclical months
X = combined[['month_sin', 'month_cos']]

# Setting up for BRR loop
time_cols = ['year', 'month', 'month_sin', 'month_cos']
all_vars  = [col for col in combined.columns if col not in time_cols]
all_vars  = [col for col in all_vars if combined[col].dtype != 'datetime64[ns]']

# Train is 2019-2023 (~70%) and test is 2024-2025 (~30%)
train_mask = (combined['year'] >= 2019) & (combined['year'] <= 2023)
test_mask  = (combined['year'] >= 2024) & (combined['year'] <= 2025)

# Robustness of alpha_1 vs lambda_1
# Ranges to test
alpha_1_vals  = [1e-10, 1e-6, 1e-4, 1e-1, 1.0, 10.0]
lambda_1_vals = [1e-10, 1e-6, 1e-4, 1e-1, 1.0, 10.0]

# BRR loop
sensitivity_prior = {}
for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    grid = np.zeros((len(alpha_1_vals), len(lambda_1_vals)))
    for i, a1 in enumerate(alpha_1_vals):
        for j, l1 in enumerate(lambda_1_vals):
            # BRR
            clf = linear_model.BayesianRidge(
                alpha_1=a1, alpha_2=DEFAULT_ALPHA_2,
                lambda_1=l1, lambda_2=DEFAULT_LAMBDA_2
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            # MAPE
            mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100
            grid[i, j] = mape
    sensitivity_prior[var] = grid
    print(f"[Prior grid] {var} done")

# Robustness of number of iterations vs tolerance
# Range of vals
n_iter_vals = [100, 200, 300, 500]
tol_vals    = [1e-3, 1e-2, 1e-1, 0.5]

# BRR loop
sensitivity_conv = {}
for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    grid = np.zeros((len(n_iter_vals), len(tol_vals)))
    for i, n in enumerate(n_iter_vals):
        for j, t in enumerate(tol_vals):
            clf = linear_model.BayesianRidge(max_iter=n, tol=t)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100
            grid[i, j] = mape
    sensitivity_conv[var] = grid
    print(f"[Conv grid] {var} done")

# Plot results on heatmap
n_vars = len(all_vars)
fig, axes = plt.subplots(2, n_vars, figsize=(8 * n_vars, 14), facecolor=LIGHT)

# Tick marks
prior_ticks  = ['1e-10', '1e-7', '1e-4', '1e-1', '1', '10']
conv_x_ticks = ['1e-3', '1e-2', '1e-1', '0.5']
conv_y_ticks = ['100', '200', '300', '500']

# Want to clearly show default vals in plot
default_a1_idx  = min(range(len(alpha_1_vals)),  key=lambda i: abs(alpha_1_vals[i]  - DEFAULT_ALPHA_1))
default_l1_idx  = min(range(len(lambda_1_vals)), key=lambda i: abs(lambda_1_vals[i] - DEFAULT_LAMBDA_1))
default_n_idx   = min(range(len(n_iter_vals)),   key=lambda i: abs(n_iter_vals[i]   - DEFAULT_N_ITER))
default_tol_idx = min(range(len(tol_vals)),      key=lambda i: abs(tol_vals[i]      - DEFAULT_TOL))

for col, var in enumerate(all_vars):
    # alpha_1 and lambda_1 for first row
    ax = axes[0, col]
    grid = sensitivity_prior[var]
    grid_rounded = np.round(grid, 1)
    sns.heatmap(grid_rounded, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
                xticklabels=prior_ticks, yticklabels=prior_ticks,
                linewidths=0.5, linecolor='#E0E0E0', 
                annot_kws={'size': 11},
                cbar_kws={'label': 'MAPE (%)', 'shrink': 0.8})

    # Put box around default vals
    ax.add_patch(plt.Rectangle(
        (default_l1_idx, default_a1_idx), 1, 1,
        fill=False, edgecolor='#2C3E6B', linewidth=2.5))

    ax.set_title(VAR_LABELS.get(var, var), fontweight='bold', fontsize=15)
    ax.set_xlabel('lambda_1', fontsize=14)
    ax.set_ylabel('alpha_1', fontsize=14)
    ax.tick_params(labelsize=12)
    default_mape = grid[default_a1_idx, default_l1_idx]
    ax.text(0.5, -0.18,
            f'Default: alpha_1={DEFAULT_ALPHA_1:.0e}, lambda_1={DEFAULT_LAMBDA_1:.0e}'
            f'\nMAPE={default_mape:.2f}%',
            transform=ax.transAxes, fontsize=10,
            ha='center', color='#333333')

    # maximum iterations and tolderance for second row
    ax = axes[1, col]
    grid = sensitivity_conv[var]
    grid_rounded = np.round(grid, 1)
    sns.heatmap(grid_rounded, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
                xticklabels=conv_x_ticks, yticklabels=conv_y_ticks,
                linewidths=0.5, linecolor='#E0E0E0',
                annot_kws={'size': 11},
                cbar_kws={'label': 'MAPE (%)', 'shrink': 0.8})

    # Put box around default vals
    ax.add_patch(plt.Rectangle(
        (default_tol_idx, default_n_idx), 1, 1,
        fill=False, edgecolor='#2C3E6B', linewidth=2.5))

    ax.set_xlabel('tol', fontsize=14)
    ax.set_ylabel('max_iter', fontsize=14)
    ax.tick_params(labelsize=12)
    default_mape = grid[default_n_idx, default_tol_idx]
    ax.text(0.5, -0.18,
            f'Default: max_iter={DEFAULT_N_ITER}, tol={DEFAULT_TOL:.0e}'
            f'\nMAPE={default_mape:.2f}%',
            transform=ax.transAxes, fontsize=10,
            ha='center', color='#333333')

# Labels
axes[0, 0].annotate('Prior Hyperparameters\n(alpha_1 vs lambda_1)',
                    xy=(0, 0.5), xytext=(-0.3, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    fontsize=12, fontweight='bold',
                    ha='center', va='center', rotation=90)
axes[1, 0].annotate('Convergence Hyperparameters\n(max_iter vs tol)',
                    xy=(0, 0.5), xytext=(-0.3, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    fontsize=12, fontweight='bold',
                    ha='center', va='center', rotation=90)

# Title
fig.suptitle(
    'BRR Hyperparameter Sensitivity — MAPE (%)\n'
    'Blue border = sklearn default parameter values',
    fontsize=16, fontweight='bold', y=1.01, color='#1A1A2E'
)
fig.text(0.5, -0.01,
         'Lower MAPE = better  |  Test period: 2024–2025  |  '
         'Cyclical month encoding (sin/cos)',
         ha='center', fontsize=10, color='#777777', style='italic')

plt.tight_layout()
plt.savefig('brr_hyperparameter_sensitivity.png', dpi=180,
            bbox_inches='tight', facecolor=LIGHT)
plt.show()

In [ ]:
# Train vs Test MAE Scatter (two rows)

# Setting up plot
figB, axesB = plt.subplots(2, n_vars, figsize=(6 * n_vars, 12), facecolor=LIGHT)
figB.subplots_adjust(hspace=0.45)

alpha_colors = plt.cm.viridis(np.linspace(0, 1, len(alpha_1_vals)))
niter_colors = plt.cm.plasma(np.linspace(0, 1, len(n_iter_vals)))

# BRR loop, finding alpha v lambda range MAE
alpha_train_mae = {var: [] for var in all_vars}
alpha_test_mae  = {var: [] for var in all_vars}

for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    for a1 in alpha_1_vals:
        clf = linear_model.BayesianRidge(
            alpha_1=a1, alpha_2=DEFAULT_ALPHA_2,
            lambda_1=DEFAULT_LAMBDA_1, lambda_2=DEFAULT_LAMBDA_2
        )
        clf.fit(X_train, y_train)
        alpha_train_mae[var].append(mean_absolute_error(y_train, clf.predict(X_train)))
        alpha_test_mae[var].append(mean_absolute_error(y_test,  clf.predict(X_test)))
        
# BRR loop, finding max iterations v tolerance MAE
niter_train_mae = {var: [] for var in all_vars}
niter_test_mae  = {var: [] for var in all_vars}

for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    for n in n_iter_vals:
        clf = linear_model.BayesianRidge(max_iter=n, tol=DEFAULT_TOL)
        clf.fit(X_train, y_train)
        niter_train_mae[var].append(mean_absolute_error(y_train, clf.predict(X_train)))
        niter_test_mae[var].append(mean_absolute_error(y_test,  clf.predict(X_test)))

for col, var in enumerate(all_vars):
    all_train    = alpha_train_mae[var] + niter_train_mae[var]
    all_test     = alpha_test_mae[var]  + niter_test_mae[var]
    all_combined = all_train + all_test
    # Jitter to see some separation in zoomed out plot
    jitter_scale = (max(all_combined) - min(all_combined)) * 0.05

    for row in range(2):
        ax = axesB[row, col]
        # alpha_1 dynamic range MAE points 
        for k, a1 in enumerate(alpha_1_vals):
            # Again jitter is only for visualization
            jx = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            jy = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            ax.scatter(alpha_train_mae[var][k] + jx,
                       alpha_test_mae[var][k]  + jy,
                       color=alpha_colors[k], s=120, zorder=5,
                       label=f'α₁={a1:.0e}', marker='o')
            if k == default_a1_idx:
                ax.scatter(alpha_train_mae[var][k] + jx,
                           alpha_test_mae[var][k]  + jy,
                           s=300, facecolors='none', edgecolors=NAVY,
                           linewidths=2.5, zorder=6)

        # number itrations dynamic range MAE points
        for k, n in enumerate(n_iter_vals):
            # jitter for visualization
            jx = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            jy = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            ax.scatter(niter_train_mae[var][k] + jx,
                       niter_test_mae[var][k]  + jy,
                       color=niter_colors[k], s=120, zorder=5,
                       label=f'n_iter={n}', marker='s')
            if k == default_n_idx:
                ax.scatter(niter_train_mae[var][k] + jx,
                           niter_test_mae[var][k]  + jy,
                           s=300, facecolors='none', edgecolors=RED,
                           linewidths=2.5, zorder=6)
    
        
        
        #  Axis labels
        ax.set_title(VAR_LABELS.get(var, var), fontweight='bold', fontsize=13)
        ax.set_xlabel('Train MAE', fontsize=11)
        ax.set_ylabel('Test MAE', fontsize=11)
        
        # Zoomed out plot is row 1
        if row == 0:
            true_min = min(all_combined) * 0.95
            true_max = max(all_combined) * 1.05
            # Plot diagonal line for ideal test/train split
            ax.plot([true_min, true_max], [true_min, true_max],
                    color='#AAAAAA', linewidth=1.0, linestyle='--',
                    label='Train = Test', zorder=1)
            ax.set_xlim(true_min, true_max)
            ax.set_ylim(true_min, true_max)
            ax.legend(fontsize=7, framealpha=0.85, edgecolor='#CCCCCC',
                      loc='upper left', ncol=1)
            # Text to explain
            ax.text(0.98, 0.02,
                    'above line = test error > train error (overfitting)\n'
                    'below line = test error < train error (underfitting)',
                    transform=ax.transAxes, fontsize=7,
                    ha='right', va='bottom', color='#555555',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white',
                              ec='#CCCCCC', alpha=0.8))
            if col == 0:
                ax.set_ylabel('Test MAE', fontsize=11)

        # Zoomed in plot is row 2
        else:
            x_min, x_max = min(all_train), max(all_train)
            y_min, y_max = min(all_test),  max(all_test)
            x_pad = (x_max - x_min) * 0.1 if x_max != x_min else abs(x_min) * 0.001
            y_pad = (y_max - y_min) * 0.1 if y_max != y_min else abs(y_min) * 0.001
            ax.set_xlim(x_min - x_pad, x_max + x_pad)
            ax.set_ylim(y_min - y_pad, y_max + y_pad)
            ax.legend(fontsize=7, framealpha=0.85, edgecolor='#CCCCCC',
                      loc='upper left', ncol=1)
            if col == 0:
                ax.set_ylabel('Test MAE', fontsize=11)

# Row labels 
axesB[0, 0].annotate('Zoomed Out\n(diagonal = ideal Train=Test)',
                     xy=(0, 0.5), xytext=(-0.3, 0.5),
                     xycoords='axes fraction', textcoords='axes fraction',
                     fontsize=11, fontweight='bold',
                     ha='center', va='center', rotation=90)
axesB[1, 0].annotate('Zoomed In\n(point separation)',
                     xy=(0, 0.5), xytext=(-0.3, 0.5),
                     xycoords='axes fraction', textcoords='axes fraction',
                     fontsize=11, fontweight='bold',
                     ha='center', va='center', rotation=90)


# Title
figB.suptitle(
    'Train vs Test MAE — Hyperparameter Scatter\n',
    fontsize=16, fontweight='bold', y=1.01, color='#1A1A2E'
)
figB.text(0.5, -0.01,
          'Jitter applied for visibility',
          ha='center', fontsize=10, color='#777777', style='italic')

plt.tight_layout()
plt.savefig('figB_train_test_scatter.png', dpi=300,
            bbox_inches='tight', facecolor=LIGHT)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model
from sklearn.metrics import mean_absolute_error

# Fonts and tick marks and such
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Georgia', 'Times New Roman', 'DejaVu Serif'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'grid.linewidth': 0.6,
    'figure.facecolor': '#FAFAF8',
    'axes.facecolor': '#FAFAF8',
    'axes.labelsize'  : 14,
    'axes.titlesize'  : 15,
    'xtick.labelsize' : 12,
    'ytick.labelsize' : 12,
    'legend.fontsize' : 12,
    'figure.titlesize': 16,
})

# Colors
LIGHT = '#FAFAF8'
LIGHT = '#FAFAF8'
NAVY = '#2C3E6B'
RED  = '#C0392B'

# Labels for variables (full name and measurement scale)
VAR_LABELS = {
    'tasmax'            : 'Max Temp (°C)',
    'tasmin'            : 'Min Temp (°C)',
    'rsds'              : 'Solar Radiation (W/m²)',
    'sfcWind'           : 'Wind Speed (m/s)',
    'ps'                : 'Surface Pressure (Pa)',
    'pr'                : 'Precipitation (kg/m²/s)',
    'Anxiety Disorders' : 'Anxiety Disorders',
}

# Default Scikit-learn values for params
DEFAULT_ALPHA_1  = 1e-6
DEFAULT_ALPHA_2  = 1e-6
DEFAULT_LAMBDA_1 = 1e-6
DEFAULT_LAMBDA_2 = 1e-6
DEFAULT_N_ITER   = 300
DEFAULT_TOL      = 1e-3

# Load and prep weather data
weather = pd.read_csv('San_Francisco_climate_v2.csv')
# Split date (called time) into year and month
weather['time'] = pd.to_datetime(weather['time'])
weather['year']  = weather['time'].dt.year
weather['month'] = weather['time'].dt.month
weather = weather.groupby(['year', 'month']).mean().reset_index()
weather = weather.drop(columns=['time'], errors='ignore')
weather = weather.drop(columns=['time'], errors='ignore')
# Make months cyclical so that the model understands the order
# of months e.g., Jan comes after Dec and Feb comes after Jan
# (aka they aren't just numbers 1-12 that repeat anymore)
weather['month_sin'] = np.sin(2 * np.pi * weather['month'] / 12)
weather['month_cos'] = np.cos(2 * np.pi * weather['month'] / 12)

# Load and prep mental health data to get anxiety
mental = pd.read_csv('mental_health_ED.csv')
ANXIETY_COL = 'Anxiety Disorders'
# Only want the total values of anxiety for each month (All)
mental = mental[mental['demographics_values'] == 'All']
mental = mental[mental['condition'] == 'Anxiety Disorders']
mental['month_end'] = pd.to_datetime(mental['month_end'])
# Pivot so that it is in same format as weather data
mental_pivot = mental.pivot_table(
    index='month_end',
    values='rate_per_100000_visits'
).reset_index()
mental_pivot.columns = ['month_end', ANXIETY_COL]
# Split date (called month_end) into year and month
mental_pivot['month_end'] = pd.to_datetime(mental_pivot['month_end'])
mental_pivot['year']  = mental_pivot['month_end'].dt.year
mental_pivot['month'] = mental_pivot['month_end'].dt.month
mental_pivot = mental_pivot.drop(columns='month_end')

# Merge into one data set
combined = weather.merge(mental_pivot, on=['year', 'month'], how='inner')

# Features are cyclical months
X = combined[['month_sin', 'month_cos']]

# Setting up for BRR loop
time_cols = ['year', 'month', 'month_sin', 'month_cos']
all_vars  = [col for col in combined.columns if col not in time_cols]
all_vars  = [col for col in all_vars if combined[col].dtype != 'datetime64[ns]']

# Train is 2019-2023 (~70%) and test is 2024-2025 (~30%)
train_mask = (combined['year'] >= 2019) & (combined['year'] <= 2023)
test_mask  = (combined['year'] >= 2024) & (combined['year'] <= 2025)

# Robustness of alpha_1 vs lambda_1
# Ranges to test
alpha_1_vals  = [1e-10, 1e-6, 1e-4, 1e-1, 1.0, 10.0]
lambda_1_vals = [1e-10, 1e-6, 1e-4, 1e-1, 1.0, 10.0]

# BRR loop
sensitivity_prior = {}
for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    grid = np.zeros((len(alpha_1_vals), len(lambda_1_vals)))
    for i, a1 in enumerate(alpha_1_vals):
        for j, l1 in enumerate(lambda_1_vals):
            # BRR
            clf = linear_model.BayesianRidge(
                alpha_1=a1, alpha_2=DEFAULT_ALPHA_2,
                lambda_1=l1, lambda_2=DEFAULT_LAMBDA_2
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            # MAPE
            mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100
            grid[i, j] = mape
    sensitivity_prior[var] = grid
    print(f"[Prior grid] {var} done")

# Robustness of number of iterations vs tolerance
# Range of vals
n_iter_vals = [100, 200, 300, 500]
tol_vals    = [1e-3, 1e-2, 1e-1, 0.5]

# BRR loop
sensitivity_conv = {}
for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    grid = np.zeros((len(n_iter_vals), len(tol_vals)))
    for i, n in enumerate(n_iter_vals):
        for j, t in enumerate(tol_vals):
            clf = linear_model.BayesianRidge(max_iter=n, tol=t)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100
            grid[i, j] = mape
    sensitivity_conv[var] = grid
    print(f"[Conv grid] {var} done")

# Plot results on heatmap
n_vars = len(all_vars)
fig, axes = plt.subplots(2, n_vars, figsize=(8 * n_vars, 14), facecolor=LIGHT)

# Tick marks
prior_ticks  = ['1e-10', '1e-7', '1e-4', '1e-1', '1', '10']
conv_x_ticks = ['1e-3', '1e-2', '1e-1', '0.5']
conv_y_ticks = ['100', '200', '300', '500']

# Want to clearly show default vals in plot
default_a1_idx  = min(range(len(alpha_1_vals)),  key=lambda i: abs(alpha_1_vals[i]  - DEFAULT_ALPHA_1))
default_l1_idx  = min(range(len(lambda_1_vals)), key=lambda i: abs(lambda_1_vals[i] - DEFAULT_LAMBDA_1))
default_n_idx   = min(range(len(n_iter_vals)),   key=lambda i: abs(n_iter_vals[i]   - DEFAULT_N_ITER))
default_tol_idx = min(range(len(tol_vals)),      key=lambda i: abs(tol_vals[i]      - DEFAULT_TOL))

for col, var in enumerate(all_vars):
    # alpha_1 and lambda_1 for first row
    ax = axes[0, col]
    grid = sensitivity_prior[var]
    grid_rounded = np.round(grid, 1)
    sns.heatmap(grid_rounded, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
                xticklabels=prior_ticks, yticklabels=prior_ticks,
                linewidths=0.5, linecolor='#E0E0E0', 
                annot_kws={'size': 11},
                cbar_kws={'label': 'MAPE (%)', 'shrink': 0.8})

    # Put box around default vals
    ax.add_patch(plt.Rectangle(
        (default_l1_idx, default_a1_idx), 1, 1,
        fill=False, edgecolor='#2C3E6B', linewidth=2.5))

    ax.set_title(VAR_LABELS.get(var, var), fontweight='bold', fontsize=15)
    ax.set_xlabel('lambda_1', fontsize=14)
    ax.set_ylabel('alpha_1', fontsize=14)
    ax.tick_params(labelsize=12)
    default_mape = grid[default_a1_idx, default_l1_idx]
    ax.text(0.5, -0.18,
            f'Default: alpha_1={DEFAULT_ALPHA_1:.0e}, lambda_1={DEFAULT_LAMBDA_1:.0e}'
            f'\nMAPE={default_mape:.2f}%',
            transform=ax.transAxes, fontsize=10,
            ha='center', color='#333333')

    # maximum iterations and tolderance for second row
    ax = axes[1, col]
    grid = sensitivity_conv[var]
    grid_rounded = np.round(grid, 1)
    sns.heatmap(grid_rounded, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
                xticklabels=conv_x_ticks, yticklabels=conv_y_ticks,
                linewidths=0.5, linecolor='#E0E0E0',
                annot_kws={'size': 11},
                cbar_kws={'label': 'MAPE (%)', 'shrink': 0.8})

    # Put box around default vals
    ax.add_patch(plt.Rectangle(
        (default_tol_idx, default_n_idx), 1, 1,
        fill=False, edgecolor='#2C3E6B', linewidth=2.5))

    ax.set_xlabel('tol', fontsize=14)
    ax.set_ylabel('max_iter', fontsize=14)
    ax.tick_params(labelsize=12)
    default_mape = grid[default_n_idx, default_tol_idx]
    ax.text(0.5, -0.18,
            f'Default: max_iter={DEFAULT_N_ITER}, tol={DEFAULT_TOL:.0e}'
            f'\nMAPE={default_mape:.2f}%',
            transform=ax.transAxes, fontsize=10,
            ha='center', color='#333333')

# Labels
axes[0, 0].annotate('Prior Hyperparameters\n(alpha_1 vs lambda_1)',
                    xy=(0, 0.5), xytext=(-0.3, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    fontsize=12, fontweight='bold',
                    ha='center', va='center', rotation=90)
axes[1, 0].annotate('Convergence Hyperparameters\n(max_iter vs tol)',
                    xy=(0, 0.5), xytext=(-0.3, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    fontsize=12, fontweight='bold',
                    ha='center', va='center', rotation=90)

# Title
fig.suptitle(
    'BRR Hyperparameter Sensitivity — MAPE (%)\n'
    'Blue border = sklearn default parameter values',
    fontsize=16, fontweight='bold', y=1.01, color='#1A1A2E'
)
fig.text(0.5, -0.01,
         'Lower MAPE = better  |  Test period: 2024–2025  |  '
         'Cyclical month encoding (sin/cos)',
         ha='center', fontsize=10, color='#777777', style='italic')

plt.tight_layout()
plt.savefig('brr_hyperparameter_sensitivity.png', dpi=180,
            bbox_inches='tight', facecolor=LIGHT)
plt.show()

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn import linear_model
# from sklearn.metrics import mean_absolute_error

# # Fonts and tick marks and such
# plt.rcParams.update({
#     'font.family': 'serif',
#     'font.serif': ['Georgia', 'Times New Roman', 'DejaVu Serif'],
#     'axes.spines.top': False,
#     'axes.spines.right': False,
#     'axes.grid': True,
#     'grid.alpha': 0.25,
#     'grid.linestyle': '--',
#     'grid.linewidth': 0.6,
#     'figure.facecolor': '#FAFAF8',
#     'axes.facecolor': '#FAFAF8',
#     'axes.labelsize'  : 14,
#     'axes.titlesize'  : 15,
#     'xtick.labelsize' : 12,
#     'ytick.labelsize' : 12,
#     'legend.fontsize' : 12,
#     'figure.titlesize': 16,
# })

# # Colors
# LIGHT = '#FAFAF8'
# NAVY = '#2C3E6B'
# RED  = '#C0392B'

# # Labels for variables (full name and measurement scale)
# VAR_LABELS = {
#     'tasmax'            : 'Max Temp (°C)',
#     'tasmin'            : 'Min Temp (°C)',
#     'rsds'              : 'Solar Radiation (W/m²)',
#     'sfcWind'           : 'Wind Speed (m/s)',
#     'ps'                : 'Surface Pressure (Pa)',
#     'pr'                : 'Precipitation (kg/m²/s)',
#     'Anxiety Disorders' : 'Anxiety Disorders',
# }

# # Default Scikit-learn values for params
# DEFAULT_ALPHA_1  = 1e-6
# DEFAULT_ALPHA_2  = 1e-6
# DEFAULT_LAMBDA_1 = 1e-6
# DEFAULT_LAMBDA_2 = 1e-6
# DEFAULT_N_ITER   = 300
# DEFAULT_TOL      = 1e-3

# # Load and prep weather data
# weather = pd.read_csv('San_Francisco_climate_v2.csv')
# # Split date (called time) into year and month
# weather['time'] = pd.to_datetime(weather['time'])
# weather['year']  = weather['time'].dt.year
# weather['month'] = weather['time'].dt.month
# weather = weather.groupby(['year', 'month']).mean().reset_index()
# weather = weather.drop(columns=['time'], errors='ignore')
# # Make months cyclical so that the model understands the order
# # of months e.g., Jan comes after Dec and Feb comes after Jan
# # (aka they aren't just numbers 1-12 that repeat anymore)
# weather['month_sin'] = np.sin(2 * np.pi * weather['month'] / 12)
# weather['month_cos'] = np.cos(2 * np.pi * weather['month'] / 12)

# # Load and prep mental health data to get anxiety
# mental = pd.read_csv('mental_health_ED.csv')
# ANXIETY_COL = 'Anxiety Disorders'
# # Only want the total values of anxiety for each month (All)
# mental = mental[mental['demographics_values'] == 'All']
# mental = mental[mental['condition'] == 'Anxiety Disorders']
# mental['month_end'] = pd.to_datetime(mental['month_end'])
# # Pivot so that it is in same format as weather data
# mental_pivot = mental.pivot_table(
#     index='month_end',
#     values='rate_per_100000_visits'
# ).reset_index()
# mental_pivot.columns = ['month_end', ANXIETY_COL]
# # Split date (called month_end) into year and month
# mental_pivot['month_end'] = pd.to_datetime(mental_pivot['month_end'])
# mental_pivot['year']  = mental_pivot['month_end'].dt.year
# mental_pivot['month'] = mental_pivot['month_end'].dt.month
# mental_pivot = mental_pivot.drop(columns='month_end')

# # Merge into one data set
# combined = weather.merge(mental_pivot, on=['year', 'month'], how='inner')

# # Features are cyclical months
# X = combined[['month_sin', 'month_cos']]

# # Setting up for BRR loop
# time_cols = ['year', 'month', 'month_sin', 'month_cos']
# all_vars  = [col for col in combined.columns if col not in time_cols]
# all_vars  = [col for col in all_vars if combined[col].dtype != 'datetime64[ns]']

# # Train is 2019-2023 (~70%) and test is 2024-2025 (~30%)
# train_mask = (combined['year'] >= 2019) & (combined['year'] <= 2023)
# test_mask  = (combined['year'] >= 2024) & (combined['year'] <= 2025)

# # ════════════════════════════════════════════════════════════════════════════
# # GRID 1 — alpha_1 vs lambda_1
# # ════════════════════════════════════════════════════════════════════════════
# alpha_1_vals  = [1e-10, 1e-6, 1e-4, 1e-1, 1.0, 10.0]
# lambda_1_vals = [1e-10, 1e-6, 1e-4, 1e-1, 1.0, 10.0]

# sensitivity_prior = {}
# for var in all_vars:
#     y = combined[var]
#     X_train, y_train = X[train_mask], y[train_mask]
#     X_test,  y_test  = X[test_mask],  y[test_mask]
#     grid = np.zeros((len(alpha_1_vals), len(lambda_1_vals)))
#     for i, a1 in enumerate(alpha_1_vals):
#         for j, l1 in enumerate(lambda_1_vals):
#             clf = linear_model.BayesianRidge(
#                 alpha_1=a1, alpha_2=DEFAULT_ALPHA_2,
#                 lambda_1=l1, lambda_2=DEFAULT_LAMBDA_2
#             )
#             clf.fit(X_train, y_train)
#             y_pred = clf.predict(X_test)
#             mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100
#             grid[i, j] = mape
#     sensitivity_prior[var] = grid
#     print(f"[Prior grid] {var} done")

# # ════════════════════════════════════════════════════════════════════════════
# # GRID 2 — n_iter vs tol
# # ════════════════════════════════════════════════════════════════════════════
# n_iter_vals = [100, 200, 300, 500]
# tol_vals    = [1e-3, 1e-2, 1e-1, 0.5]

# sensitivity_conv = {}
# for var in all_vars:
#     y = combined[var]
#     X_train, y_train = X[train_mask], y[train_mask]
#     X_test,  y_test  = X[test_mask],  y[test_mask]
#     grid = np.zeros((len(n_iter_vals), len(tol_vals)))
#     for i, n in enumerate(n_iter_vals):
#         for j, t in enumerate(tol_vals):
#             clf = linear_model.BayesianRidge(max_iter=n, tol=t)
#             clf.fit(X_train, y_train)
#             y_pred = clf.predict(X_test)
#             mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100
#             grid[i, j] = mape
#     sensitivity_conv[var] = grid
#     print(f"[Conv grid] {var} done")

# # ════════════════════════════════════════════════════════════════════════════
# # GRID 3 — Train vs Test MAE across alpha_1
# # ════════════════════════════════════════════════════════════════════════════
# alpha_train_mae = {var: [] for var in all_vars}
# alpha_test_mae  = {var: [] for var in all_vars}

# for var in all_vars:
#     y = combined[var]
#     X_train, y_train = X[train_mask], y[train_mask]
#     X_test,  y_test  = X[test_mask],  y[test_mask]
#     for a1 in alpha_1_vals:
#         clf = linear_model.BayesianRidge(
#             alpha_1=a1, alpha_2=DEFAULT_ALPHA_2,
#             lambda_1=DEFAULT_LAMBDA_1, lambda_2=DEFAULT_LAMBDA_2
#         )
#         clf.fit(X_train, y_train)
#         alpha_train_mae[var].append(mean_absolute_error(y_train, clf.predict(X_train)))
#         alpha_test_mae[var].append(mean_absolute_error(y_test,  clf.predict(X_test)))

# # ════════════════════════════════════════════════════════════════════════════
# # GRID 4 — Train vs Test MAE across n_iter
# # ════════════════════════════════════════════════════════════════════════════
# niter_train_mae = {var: [] for var in all_vars}
# niter_test_mae  = {var: [] for var in all_vars}

# for var in all_vars:
#     y = combined[var]
#     X_train, y_train = X[train_mask], y[train_mask]
#     X_test,  y_test  = X[test_mask],  y[test_mask]
#     for n in n_iter_vals:
#         clf = linear_model.BayesianRidge(max_iter=n, tol=DEFAULT_TOL)
#         clf.fit(X_train, y_train)
#         niter_train_mae[var].append(mean_absolute_error(y_train, clf.predict(X_train)))
#         niter_test_mae[var].append(mean_absolute_error(y_test,  clf.predict(X_test)))

# # find default indices
# default_a1_idx  = min(range(len(alpha_1_vals)),  key=lambda i: abs(alpha_1_vals[i]  - DEFAULT_ALPHA_1))
# default_l1_idx  = min(range(len(lambda_1_vals)), key=lambda i: abs(lambda_1_vals[i] - DEFAULT_LAMBDA_1))
# default_n_idx   = min(range(len(n_iter_vals)),   key=lambda i: abs(n_iter_vals[i]   - DEFAULT_N_ITER))
# default_tol_idx = min(range(len(tol_vals)),      key=lambda i: abs(tol_vals[i]      - DEFAULT_TOL))

# n_vars = len(all_vars)
# prior_ticks  = ['1e-10', '1e-6', '1e-4', '1e-1', '1', '10']
# conv_x_ticks = ['1e-3', '1e-2', '1e-1', '0.5']
# conv_y_ticks = ['100', '200', '300', '500']
# alpha_labels = ['1e-10', '1e-6', '1e-4', '1e-1', '1', '10']
# niter_labels = ['100', '200', '300', '500']

# # ════════════════════════════════════════════════════════════════════════════
# # FIGURE A — Heatmaps
# # ════════════════════════════════════════════════════════════════════════════
# figA, axesA = plt.subplots(2, n_vars, figsize=(8 * n_vars, 14), facecolor=LIGHT)

# for col, var in enumerate(all_vars):

#     # ── Row 1: prior sensitivity ───────────────────────────────────────────
#     ax = axesA[0, col]
#     grid = sensitivity_prior[var]
#     grid_rounded = np.round(grid, 1)
#     sns.heatmap(grid_rounded, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
#                 xticklabels=prior_ticks, yticklabels=prior_ticks,
#                 linewidths=0.5, linecolor='#E0E0E0',
#                 annot_kws={'size': 11},
#                 cbar_kws={'label': 'MAPE (%)', 'shrink': 0.8})
#     ax.add_patch(plt.Rectangle(
#         (default_l1_idx, default_a1_idx), 1, 1,
#         fill=False, edgecolor='#2C3E6B', linewidth=2.5))
#     ax.set_title(VAR_LABELS.get(var, var), fontweight='bold', fontsize=15)
#     ax.set_xlabel('lambda_1', fontsize=14)
#     ax.set_ylabel('alpha_1', fontsize=14)
#     ax.tick_params(labelsize=12)
#     default_mape = grid[default_a1_idx, default_l1_idx]
#     ax.text(0.5, -0.18,
#             f'Default: alpha_1={DEFAULT_ALPHA_1:.0e}, lambda_1={DEFAULT_LAMBDA_1:.0e}'
#             f'\nMAPE={default_mape:.2f}%',
#             transform=ax.transAxes, fontsize=10,
#             ha='center', color='#333333')

#     # ── Row 2: convergence sensitivity ────────────────────────────────────
#     ax = axesA[1, col]
#     grid = sensitivity_conv[var]
#     grid_rounded = np.round(grid, 1)
#     sns.heatmap(grid_rounded, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
#                 xticklabels=conv_x_ticks, yticklabels=conv_y_ticks,
#                 linewidths=0.5, linecolor='#E0E0E0',
#                 annot_kws={'size': 11},
#                 cbar_kws={'label': 'MAPE (%)', 'shrink': 0.8})
#     ax.add_patch(plt.Rectangle(
#         (default_tol_idx, default_n_idx), 1, 1,
#         fill=False, edgecolor='#2C3E6B', linewidth=2.5))
#     ax.set_xlabel('tol', fontsize=14)
#     ax.set_ylabel('max_iter', fontsize=14)
#     ax.tick_params(labelsize=12)
#     default_mape = grid[default_n_idx, default_tol_idx]
#     ax.text(0.5, -0.18,
#             f'Default: max_iter={DEFAULT_N_ITER}, tol={DEFAULT_TOL:.0e}'
#             f'\nMAPE={default_mape:.2f}%',
#             transform=ax.transAxes, fontsize=10,
#             ha='center', color='#333333')

# axesA[0, 0].annotate('Prior Hyperparameters\n(alpha_1 vs lambda_1)',
#                      xy=(0, 0.5), xytext=(-0.3, 0.5),
#                      xycoords='axes fraction', textcoords='axes fraction',
#                      fontsize=12, fontweight='bold',
#                      ha='center', va='center', rotation=90)
# axesA[1, 0].annotate('Convergence Hyperparameters\n(max_iter vs tol)',
#                      xy=(0, 0.5), xytext=(-0.3, 0.5),
#                      xycoords='axes fraction', textcoords='axes fraction',
#                      fontsize=12, fontweight='bold',
#                      ha='center', va='center', rotation=90)

# figA.suptitle(
#     'BRR Hyperparameter Sensitivity — MAPE (%)\n'
#     'Blue border = sklearn default parameter values',
#     fontsize=16, fontweight='bold', y=1.01, color='#1A1A2E'
# )
# # figA.text(0.5, -0.01,
# #           ha='center', fontsize=10, color='#777777', style='italic')

# plt.tight_layout()
# plt.savefig('figA_hyperparameter_sensitivity.png', dpi=180,
#             bbox_inches='tight', facecolor=LIGHT)
# plt.show()

# # # ════════════════════════════════════════════════════════════════════════════
# # # FIGURE B — Train vs Test MAE Scatter
# # # ════════════════════════════════════════════════════════════════════════════
# # figB, axesB = plt.subplots(1, n_vars, figsize=(6 * n_vars, 6), facecolor=LIGHT)
# # figB.subplots_adjust(hspace=0.5)

# # # color each point by parameter value
# # alpha_colors = plt.cm.viridis(np.linspace(0, 1, len(alpha_1_vals)))
# # niter_colors = plt.cm.plasma(np.linspace(0, 1, len(n_iter_vals)))

# # for col, var in enumerate(all_vars):
# #     ax = axesB[col]
    
# #     jitter_scale = (max(all_combined) - min(all_combined)) * 0.05

# #     # ── alpha_1 points ────────────────────────────────────────────────────
# #     for k, a1 in enumerate(alpha_1_vals):
# #         ax.scatter(alpha_train_mae[var][k], alpha_test_mae[var][k],
# #                    color=alpha_colors[k], s=120, zorder=5,
# #                    label=f'α₁={a1:.0e}',
# #                    marker='o')
# #         # highlight default
# #         if k == default_a1_idx:
# #             ax.scatter(alpha_train_mae[var][k], alpha_test_mae[var][k],
# #                        s=300, facecolors='none', edgecolors=NAVY,
# #                        linewidths=2.5, zorder=6)

 

# #     # ── n_iter points ─────────────────────────────────────────────────────
# #     for k, n in enumerate(n_iter_vals):
# #         ax.scatter(niter_train_mae[var][k], niter_test_mae[var][k],
# #                    color=niter_colors[k], s=120, zorder=5,
# #                    label=f'n_iter={n}',
# #                    marker='s')
# #         # highlight default
# #         if k == default_n_idx:
# #             ax.scatter(niter_train_mae[var][k], niter_test_mae[var][k],
# #                        s=300, facecolors='none', edgecolors=RED,
# #                        linewidths=2.5, zorder=6)

 

# #     # ── diagonal line (perfect train=test) ────────────────────────────────
# #     all_train    = alpha_train_mae[var] + niter_train_mae[var]
# #     all_test     = alpha_test_mae[var]  + niter_test_mae[var]
# #     all_combined = all_train + all_test
# #     true_min = min(all_combined) * 0.9995
# #     true_max = max(all_combined) * 1.0005
# #     ax.plot([true_min, true_max], [true_min, true_max],
# #             color='#AAAAAA', linewidth=1.0, linestyle='--',
# #             label='Train = Test', zorder=1)
# #     ax.set_xlim(true_min, true_max)
# #     ax.set_ylim(true_min, true_max)



# #     ax.set_title(VAR_LABELS.get(var, var), fontweight='bold', fontsize=13)
# #     ax.set_xlabel('Train MAE', fontsize=11)
# #     ax.set_ylabel('Test MAE', fontsize=11)
# #     ax.legend(fontsize=7, framealpha=0.85, edgecolor='#CCCCCC',
# #               loc='upper left', ncol=1)


# # figB.text(0.5, -0.01,
# #           'Test period: 2024–2025',
# #           ha='center', fontsize=10, color='#777777', style='italic')

# # plt.tight_layout()
# # plt.savefig('figB_train_test_scatter.png', dpi=180,
# #             bbox_inches='tight', facecolor=LIGHT)
# # plt.show()

In [ ]:
# Train vs Test MAE Scatter (two rows)

# Setting up plot
figB, axesB = plt.subplots(2, n_vars, figsize=(6 * n_vars, 12), facecolor=LIGHT)
figB.subplots_adjust(hspace=0.45)

alpha_colors = plt.cm.viridis(np.linspace(0, 1, len(alpha_1_vals)))
niter_colors = plt.cm.plasma(np.linspace(0, 1, len(n_iter_vals)))

# BRR loop, finding alpha v lambda range MAE
alpha_train_mae = {var: [] for var in all_vars}
alpha_test_mae  = {var: [] for var in all_vars}

for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    for a1 in alpha_1_vals:
        clf = linear_model.BayesianRidge(
            alpha_1=a1, alpha_2=DEFAULT_ALPHA_2,
            lambda_1=DEFAULT_LAMBDA_1, lambda_2=DEFAULT_LAMBDA_2
        )
        clf.fit(X_train, y_train)
        alpha_train_mae[var].append(mean_absolute_error(y_train, clf.predict(X_train)))
        alpha_test_mae[var].append(mean_absolute_error(y_test,  clf.predict(X_test)))
        
# BRR loop, finding max iterations v tolerance MAE
niter_train_mae = {var: [] for var in all_vars}
niter_test_mae  = {var: [] for var in all_vars}

for var in all_vars:
    y = combined[var]
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    for n in n_iter_vals:
        clf = linear_model.BayesianRidge(max_iter=n, tol=DEFAULT_TOL)
        clf.fit(X_train, y_train)
        niter_train_mae[var].append(mean_absolute_error(y_train, clf.predict(X_train)))
        niter_test_mae[var].append(mean_absolute_error(y_test,  clf.predict(X_test)))

for col, var in enumerate(all_vars):
    all_train    = alpha_train_mae[var] + niter_train_mae[var]
    all_test     = alpha_test_mae[var]  + niter_test_mae[var]
    all_combined = all_train + all_test
    # Jitter to see some separation in zoomed out plot
    jitter_scale = (max(all_combined) - min(all_combined)) * 0.05

    for row in range(2):
        ax = axesB[row, col]
        # alpha_1 dynamic range MAE points 
        for k, a1 in enumerate(alpha_1_vals):
            # Again jitter is only for visualization
            jx = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            jy = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            ax.scatter(alpha_train_mae[var][k] + jx,
                       alpha_test_mae[var][k]  + jy,
                       color=alpha_colors[k], s=120, zorder=5,
                       label=f'α₁={a1:.0e}', marker='o')
            if k == default_a1_idx:
                ax.scatter(alpha_train_mae[var][k] + jx,
                           alpha_test_mae[var][k]  + jy,
                           s=300, facecolors='none', edgecolors=NAVY,
                           linewidths=2.5, zorder=6)

        # number itrations dynamic range MAE points
        for k, n in enumerate(n_iter_vals):
            # jitter for visualization
            jx = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            jy = np.random.uniform(-jitter_scale, jitter_scale) if row == 0 else 0
            ax.scatter(niter_train_mae[var][k] + jx,
                       niter_test_mae[var][k]  + jy,
                       color=niter_colors[k], s=120, zorder=5,
                       label=f'n_iter={n}', marker='s')
            if k == default_n_idx:
                ax.scatter(niter_train_mae[var][k] + jx,
                           niter_test_mae[var][k]  + jy,
                           s=300, facecolors='none', edgecolors=RED,
                           linewidths=2.5, zorder=6)
    
        
        
        #  Axis labels
        ax.set_title(VAR_LABELS.get(var, var), fontweight='bold', fontsize=13)
        ax.set_xlabel('Train MAE', fontsize=11)
        ax.set_ylabel('Test MAE', fontsize=11)
        
        # Zoomed out plot is row 1
        if row == 0:
            true_min = min(all_combined) * 0.95
            true_max = max(all_combined) * 1.05
            # Plot diagonal line for ideal test/train split
            ax.plot([true_min, true_max], [true_min, true_max],
                    color='#AAAAAA', linewidth=1.0, linestyle='--',
                    label='Train = Test', zorder=1)
            ax.set_xlim(true_min, true_max)
            ax.set_ylim(true_min, true_max)
            ax.legend(fontsize=7, framealpha=0.85, edgecolor='#CCCCCC',
                      loc='upper left', ncol=1)
            # Text to explain
            ax.text(0.98, 0.02,
                    'above line = test error > train error (overfitting)\n'
                    'below line = test error < train error (underfitting)',
                    transform=ax.transAxes, fontsize=7,
                    ha='right', va='bottom', color='#555555',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white',
                              ec='#CCCCCC', alpha=0.8))
            if col == 0:
                ax.set_ylabel('Test MAE', fontsize=11)

        # Zoomed in plot is row 2
        else:
            x_min, x_max = min(all_train), max(all_train)
            y_min, y_max = min(all_test),  max(all_test)
            x_pad = (x_max - x_min) * 0.1 if x_max != x_min else abs(x_min) * 0.001
            y_pad = (y_max - y_min) * 0.1 if y_max != y_min else abs(y_min) * 0.001
            ax.set_xlim(x_min - x_pad, x_max + x_pad)
            ax.set_ylim(y_min - y_pad, y_max + y_pad)
            ax.legend(fontsize=7, framealpha=0.85, edgecolor='#CCCCCC',
                      loc='upper left', ncol=1)
            if col == 0:
                ax.set_ylabel('Test MAE', fontsize=11)

# Row labels 
axesB[0, 0].annotate('Zoomed Out\n(diagonal = ideal Train=Test)',
                     xy=(0, 0.5), xytext=(-0.3, 0.5),
                     xycoords='axes fraction', textcoords='axes fraction',
                     fontsize=11, fontweight='bold',
                     ha='center', va='center', rotation=90)
axesB[1, 0].annotate('Zoomed In\n(point separation)',
                     xy=(0, 0.5), xytext=(-0.3, 0.5),
                     xycoords='axes fraction', textcoords='axes fraction',
                     fontsize=11, fontweight='bold',
                     ha='center', va='center', rotation=90)


# Title
figB.suptitle(
    'Train vs Test MAE — Hyperparameter Scatter\n',
    fontsize=16, fontweight='bold', y=1.01, color='#1A1A2E'
)
figB.text(0.5, -0.01,
          'Jitter applied for visibility',
          ha='center', fontsize=10, color='#777777', style='italic')

plt.tight_layout()
plt.savefig('figB_train_test_scatter.png', dpi=300,
            bbox_inches='tight', facecolor=LIGHT)
plt.show()